# Lekcija 13 - Memorija agenta s Cognee grafovima znanja


## Postavljanje

Ovaj bilježnik prikazuje kako izgraditi inteligentnog **asistenta za kodiranje** s perzistentnim memorijom koristeći [**Cognee**](https://www.cognee.ai/) grafove znanja i **Microsoft Agent Framework** (MAF).

Cognee pretvara nestrukturirani tekst u strukturirani, upitni graf znanja potpomognut vektorskim ugradbama — dajući vašem agentu bogatu, odnosno osviještenu dugoročnu memoriju.

### Ono što ćete naučiti
1. **Izgraditi Grafove Znanja**: Pretvoriti profile programera i najbolje prakse u strukturirano, upitno znanje.
2. **Integrirati Cognee s MAF-om**: Koristiti `@tool` funkcije da MAF agent može upitavati Cogneeov graf znanja.
3. **Konverzacije Svjesne Sesije**: Održavati kontekst kroz više pitanja u istoj sesiji.
4. **Dugoročna Memorija**: Sačuvati važno znanje kroz sesije i dohvatiti ga u novim razgovorima.

### Preduvjeti
- Python 3.9+
- Redis pokrenut lokalno (`docker run -d -p 6379:6379 redis`) za upravljanje sesijama
- API ključ za LLM (npr. OpenAI) — postavite `LLM_API_KEY` u `.env`
- `CACHING=true` u `.env` (potrebno za Cognee sesije)
- Microsoft Foundry projekt s implementiranim chat modelom
- `AZURE_AI_PROJECT_ENDPOINT` i `AZURE_AI_MODEL_DEPLOYMENT_NAME` u `.env`
- Azure CLI autentifikacija (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Vrste memorije agenta

Ovaj bilježnik istražuje iste tri vrste memorije iz glavnog bilježnika Lekcije 13, ali koristi Cognee kao pozadinu za dugoročnu memoriju:

| Vrsta memorije | Mehanizam | Trajanje |
|---|---|---|
| **Radna** | `agent.create_session()` (MAF) | Jedan razgovor |
| **Kratkotrajna** | Cognee predmemorija sesije (Redis) | Jedna sesija |
| **Dugotrajna** | Cognee znanstveni graf + vektori | Trajna |

### Arhitektura memorije Cogneeja
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Pripremite Cognee pohranu


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Dio 1 — Izgradnja baze znanja

Prikupljamo tri vrste podataka kako bismo stvorili sveobuhvatnu bazu znanja za našeg pomoćnika za kodiranje:

1. **Profil programera** — osobna stručnost i tehnička pozadina
2. **Najbolje prakse Pythona** — Zen Pythona s praktičnim smjernicama
3. **Povijesni razgovori** — prethodne sesije pitanja i odgovora između programera i AI pomoćnika


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizirajte Graf znanja

Cognee može prikazati interaktivnu HTML vizualizaciju entiteta i odnosa koje je izvukao.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Obogati memoriju s Memify

`memify()` analizira graf znanja i generira inteligentna pravila — prepoznavajući obrasce, najbolje prakse i odnose između pojmova.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Dio 2 — MAF agent s Cognee alatima

Sada stvaramo MAF agenta koji može upitivati Cognee-ov graf znanja putem funkcija `@tool`. To agentu omogućuje da iskoristi punu snagu semantičkog pretraživanja svjestnog grafa, dok istovremeno održava kontekst razgovora kroz sesije.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Radna memorija sa sesijama

`AgentSession` (stvoren putem `agent.create_session()`) pruža radnu memoriju unutar sesije. Agent može referirati na ranije poruke, dok istovremeno upitava Cogneeov graf dugoročnog znanja.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nova sesija — Dugoročno pamćenje opstaje

Pokretanje nove sesije briše radnu memoriju, ali Cognee graf znanja i dalje je dostupan. Agent može dohvatiti isto dugoročno znanje u potpuno novom razgovoru.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sažetak

U ovom bilježniku ste izgradili asistenta za kodiranje koji kombinira **MAF-ovu radnu memoriju** (`agent.create_session()`) s **Cogneeovim grafom znanja dugoročnog trajanja**.

### Što ste naučili
1. **Izgradnja grafa znanja**: Cognee obrađuje nestrukturirani tekst i gradi graf + vektorsku memoriju.
2. **Obogaćivanje grafa s memify**: Izvedeni činjenice i bogatiji odnosi na vašem postojećem grafu.
3. **Integracija MAF + Cognee**: `@tool` funkcije omogućuju MAF agentu da prirodno pretražuje Cogneeov graf.
4. **Radna memorija + dugoročna memorija**: `AgentSession` (putem `agent.create_session()`) pruža kontekst sesije dok Cognee osigurava trajno znanje.
5. **Filtrirano pretraživanje s NodeSets**: Ciljajte specifične podskupove grafa znanja (npr. samo principe).

### Ključni zaključci
- **Cognee** pretvara sirovi tekst u strukturiranu, odnosno osviještenu memoriju — moćniju od ravne vektorske pohrane.
- **`@tool` funkcije** čisto povezuju MAF agente i vanjske sustave znanja.
- **`AgentSession`** (putem `agent.create_session()`) održava kontekst po razgovoru odvojen od dugotrajnog znanja.
- Isti graf znanja služi za više sesija i agenata.

### Primjene u stvarnom svijetu
- **Razvojni kopiloti**: Pregled koda, analiza incidenata, asistenti za arhitekturu
- **Kopiloti za korisnike**: Agent za podršku nad dokumentacijom proizvoda, često postavljanim pitanjima i bilješkama CRM-a
- **Interni stručni kopiloti**: Asistenti za pravila, pravne ili sigurnosne postupke koji analiziraju smjernice
- **Jedinstveni slojevi podataka**: Kombinirajte strukturirane i nestrukturirane podatke u jedan upitni graf

### Sljedeći koraci
- Eksperimentirajte s vremenskom osviještenošću u Cogneeu
- Definirajte OWL ontologiju za kvalitetu grafa specifičnu za domenu
- Dodajte petlje povratnih informacija korisnika za poboljšanje pretraživanja tijekom vremena
- Skalirajte na sustave s više agenata koji dijele isti Cognee sloj memorije


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
